# Olist Seller Analytics
Análise exploratória de vendedores.

Notebook técnico assistido por IA, registro 3. Conclusões e justificativas ficam para autoria humana. Os dados usados são os originais Olist.

In [1]:
# [IA] registro 3
from pathlib import Path
import sys, json
root = Path.cwd()
if not (root / "src").exists():
    root = root.parent
sys.path.insert(0, str(root))
from src.load_data import load_data
from src.preprocessing import preprocessar, profile
from src.analysis import seller_metrics, review_comparison, selection_report
from src.visualizations import ranking_chart, review_chart
import pandas as pd
import matplotlib.pyplot as plt
raw = load_data(root / "data/raw")
names = ["orders", "items", "products", "sellers", "reviews"]

## 1. Compreensão do negócio e dos dados
Objetivo: fornecer evidências sobre despacho e avaliações. Não há modelagem preditiva. Chaves: pedido, pedido-item, produto, vendedor. Avaliações podem se repetir por pedido.

In [2]:
# [IA] registro 3
pd.DataFrame([{ "tabela": n, "linhas": len(d), "colunas": len(d.columns), "duplicatas": int(d.duplicated().sum())} for n,d in zip(names,raw)])

,tabela,linhas,colunas,duplicatas
0,orders,99441,8,0
1,items,112650,7,0
2,products,32951,9,0
3,sellers,3095,4,0
4,reviews,99224,7,0


In [3]:
# [IA] registro 3
for name, frame in zip(names, raw):
    print(name)
    display(pd.DataFrame({"tipo_carregado": frame.dtypes.astype(str), "nulos": frame.isna().sum(), "distintos": frame.nunique()}))
display(raw[1][["price", "freight_value"]].apply(pd.to_numeric).describe())
display(pd.to_numeric(raw[4].review_score).value_counts().sort_index())

orders


,tipo_carregado,nulos,distintos
order_id,string,0,99441
customer_id,string,0,99441
order_status,string,0,8
order_purchase_timestamp,string,0,98875
order_approved_at,string,160,90733
order_delivered_carrier_date,string,1783,81018
order_delivered_customer_date,string,2965,95664
order_estimated_delivery_date,string,0,459


items


,tipo_carregado,nulos,distintos
order_id,string,0,98666
order_item_id,string,0,21
product_id,string,0,32951
seller_id,string,0,3095
shipping_limit_date,string,0,93318
price,string,0,5968
freight_value,string,0,6999


products


,tipo_carregado,nulos,distintos
product_id,string,0,32951
product_category_name,string,610,73
product_name_lenght,string,610,66
product_description_lenght,string,610,2960
product_photos_qty,string,610,19
product_weight_g,string,2,2204
product_length_cm,string,2,99
product_height_cm,string,2,102
product_width_cm,string,2,95


... (2 outputs left out)

reviews


,tipo_carregado,nulos,distintos
review_id,string,0,98410
order_id,string,0,98673
review_score,string,0,5
review_comment_title,string,87656,4527
review_comment_message,string,58247,36159
review_creation_date,string,0,636
review_answer_timestamp,string,0,98248


,price,freight_value
count,112650.0,112650.0
mean,120.653739,19.99032
std,183.633928,15.806405
min,0.85,0.0
25%,39.9,13.08
50%,74.99,16.26
75%,134.9,21.15
max,6735.0,409.68


review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: Int64

## 2. Pipeline de preparação e histórico de merges
`preprocessar` executa todas as etapas sem células manuais: validação, cópia, duplicatas, chaves, tipos, datas, categorias, textos, agregações, merges e engenharia. Itens-produtos N:1; agregação pedido-vendedor; contagens por pedido N:1; pedidos N:1; vendedores N:1; avaliação única N:1. Textos livres não são exportados.

In [4]:
# [IA] registro 1
tabela = preprocessar(*raw)
audit = tabela.attrs["audit"]
print("Tabela final:", tabela.shape)
print("Chave repetida:", tabela.duplicated(["order_id", "seller_id"]).sum())
print({k: audit[k] for k in ["reviews_reduced", "orders_without_items", "multi_seller_orders", "invalid_chronology_pairs", "eligible_dispatch", "eligible_review"]})

Tabela final: (100010, 41)
Chave repetida: 0
{'reviews_reduced': 551, 'orders_without_items': 775, 'multi_seller_orders': 1278, 'invalid_chronology_pairs': 1451, 'eligible_dispatch': 94955, 'eligible_review': 94257}


## 3. Limpeza, ausências e impacto
As medidas gerenciais preservam extremos. Datas, notas e dinheiro não são imputados. Consultar evidências antes de redigir as justificativas humanas. O CSV antes/depois registra linhas e duplicatas; alternativas_outliers quantifica remoção e clipping.

In [5]:
# [IA] registro 3
print("Datas inválidas:", audit["invalid_dates"])
display(pd.DataFrame({"nulos_finais": tabela.isna().sum(), "tipo": tabela.dtypes.astype(str)}))
display(pd.read_csv(root / "reports/antes_depois.csv"))
display(pd.read_csv(root / "reports/alternativas_outliers.csv"))

Datas inválidas: {'orders.order_purchase_timestamp': 0, 'orders.order_approved_at': 0, 'orders.order_delivered_carrier_date': 0, 'orders.order_delivered_customer_date': 0, 'orders.order_estimated_delivery_date': 0, 'items.shipping_limit_date': 0, 'reviews.review_answer_timestamp': 0}


,nulos_finais,tipo
order_id,0,string
seller_id,0,string
item_count,0,int64
sales_value,0,float64
freight_value,0,float64
shipping_limit_date,0,datetime64[ns]
shipping_limit_max,0,datetime64[ns]
deadlines_complete,0,bool
category_count,0,int64
category_main,0,string


,tabela,linhas_brutas,duplicatas_exatas,linhas_tratadas,chaves_nulas_removidas
0,orders,99441,0,99441,0
1,items,112650,0,112650,0
2,products,32951,0,32951,0
3,sellers,3095,0,3095,0
4,reviews,99224,0,99224,0


,alternativa,linhas,media_dias,taxa_atraso
0,preservar,94955,2.870444,0.092349
1,excluir_outliers,89564,2.235676,0.056585
2,limitar_IQR_apenas_demonstracao,94955,2.549037,0.092349


## 4. Transformações e seleção
One-hot de status com esquema fixo; MinMax e Z-score de sales_value; clipping IQR somente demonstrativo. Atributos originais ficam preservados. Seleção semântica, variância populacional <= 1e-12 e inspeção de correlações >= 0,95 em módulo.

In [6]:
# [IA] registro 1
display(tabela[["sales_value", "demo_sales_minmax", "demo_sales_zscore", "demo_status_delivered", "demo_status_shipped", "demo_status_other"]].head())
selection, correlations = selection_report(tabela)
print(json.dumps(selection, ensure_ascii=False, indent=2))
display(correlations)

,sales_value,demo_sales_minmax,demo_sales_zscore,demo_status_delivered,demo_status_shipped,demo_status_other
0,58.90,0.004319,-0.368804,1,0,0
1,239.90,0.017788,0.498092,1,0,0
2,199.00,0.014744,0.302203,1,0,0
3,12.99,0.000903,-0.588689,1,0,0
4,199.90,0.014811,0.306513,1,0,0


{
  "low_variance": [],
  "high_correlation": [
    {
      "a": "sales_value",
      "b": "demo_sales_minmax",
      "correlation": 0.9999999999999977
    },
    {
      "a": "sales_value",
      "b": "demo_sales_zscore",
      "correlation": 0.9999999999999962
    },
    {
      "a": "demo_sales_minmax",
      "b": "demo_sales_zscore",
      "correlation": 1.0000000000000036
    }
  ],
  "selected_analysis": [
    "item_count",
    "sales_value",
    "freight_value",
    "approval_to_carrier_days",
    "dispatch_delay_days",
    "transport_days",
    "delivery_days",
    "review_score",
    "freight_share"
  ],
  "rule": "Filtro semântico: retirar IDs, textos e transformações redundantes; variância <= 1e-12; correlação absoluta >= 0.95 para inspeção, sem excluir métricas gerenciais."
}


,item_count,sales_value,freight_value,approval_to_carrier_days,dispatch_delay_days,transport_days,delivery_days,review_score,freight_share,demo_sales_minmax,demo_sales_zscore,demo_status_delivered,demo_status_shipped,demo_status_other
item_count,1.000000,0.146340,0.419604,0.037369,0.009868,-0.027095,-0.009017,-0.090255,0.076791,0.146340,0.146340,0.001993,-0.011782,0.009096
sales_value,0.146340,1.000000,0.408298,0.073439,0.013183,0.032049,0.058753,-0.030429,-0.412805,1.000000,1.000000,-0.023680,0.000067,0.033425
freight_value,0.419604,0.408298,1.000000,0.092679,0.008634,0.148503,0.173418,-0.074128,0.121072,0.408298,0.408298,-0.013781,0.006514,0.012903
approval_to_carrier_days,0.037369,0.073439,0.092679,1.000000,0.390506,0.029422,0.388918,-0.149675,-0.024495,0.073439,0.073439,-0.014421,0.014106,0.003019
dispatch_delay_days,0.009868,0.013183,0.008634,0.390506,1.000000,-0.002687,0.164487,-0.062896,-0.003252,0.013183,0.013183,0.008027,-0.008027,NaN
transport_days,-0.027095,0.032049,0.148503,0.029422,-0.002687,1.000000,0.926546,-0.290823,0.102972,0.032049,0.032049,-0.003634,NaN,0.003634
delivery_days,-0.009017,0.058753,0.173418,0.388918,0.164487,0.926546,1.000000,-0.322824,0.088169,0.058753,0.058753,-0.006463,NaN,0.006463
review_score,-0.090255,-0.030429,-0.074128,-0.149675,-0.062896,-0.290823,-0.322824,1.000000,-0.021981,-0.030429,-0.030429,0.254938,-0.159251,-0.199161
freight_share,0.076791,-0.412805,0.121072,-0.024495,-0.003252,0.102972,0.088169,-0.021981,1.000000,-0.412805,-0.412805,0.008280,0.009041,-0.020855
demo_sales_minmax,0.146340,1.000000,0.408298,0.073439,0.013183,0.032049,0.058753,-0.030429,-0.412805,1.000000,1.000000,-0.023680,0.000067,0.033425


## 5. Engenharia de atributos
Técnicas distintas: diferenças temporais; indicadores condicionais; agregação de itens/pedidos por grupo; extração de mês e dia da semana; razão entre frete e valor. A comparação individual exclui pedidos multivendedor observados.

In [7]:
# [IA] registro 1
display(tabela[["approval_to_carrier_days", "dispatch_delay_days", "dispatch_late", "transport_days", "delivery_late", "item_count", "purchase_month", "purchase_weekday", "freight_share"]].describe(include="all"))

,approval_to_carrier_days,dispatch_delay_days,dispatch_late,transport_days,delivery_late,item_count,purchase_month,purchase_weekday,freight_share
count,97588.000000,97503.000000,97503.0,97774.000000,97817.0,100010.000000,100010,100010.0,100010.000000
unique,NaN,NaN,<NA>,NaN,<NA>,NaN,24,<NA>,NaN
top,NaN,NaN,<NA>,NaN,<NA>,NaN,2017-11,<NA>,NaN
freq,NaN,NaN,<NA>,NaN,<NA>,NaN,7548,<NA>,NaN
mean,2.846690,-3.340230,0.091125,9.301073,0.066941,1.126387,NaN,2.759504,0.208975
std,3.482710,6.031126,0.287789,8.725061,0.249922,0.511118,NaN,1.968179,0.126647
min,0.000174,-1054.244525,0.0,0.000000,0.0,1.000000,NaN,0.0,0.000000
25%,0.899355,-5.210637,0.0,4.092376,0.0,1.000000,NaN,1.0,0.116038
50%,1.845058,-3.230440,0.0,7.092199,0.0,1.000000,NaN,3.0,0.183256
75%,3.603374,-1.567992,0.0,12.007098,0.0,1.000000,NaN,4.0,0.276513


## 6. Evidências para as perguntas do escopo
As tabelas abaixo são resultados calculados. A equipe deve produzir as respostas de negócio e conclusões; associação não implica causalidade.

In [8]:
# [IA] registro 2
metrics = seller_metrics(tabela)
display(metrics[metrics.eligible_orders.ge(30)].head(15))
comparison = review_comparison(tabela)
display(comparison)
display(tabela[tabela.eligible_dispatch].groupby("purchase_month").dispatch_late.agg(["mean", "count"]))
display(tabela[tabela.eligible_review].groupby(["dispatch_late", "delivery_late"], dropna=False).review_score.agg(["mean", "count"]))

,seller_id,observed_orders,sales_value,eligible_orders,late_orders,mean_delay_days,median_dispatch_days,review_count,mean_score,negative_count,late_rate,negative_rate,late_ci_low,late_ci_high
863,48efc9d94a9834137efd9ea76b065a38,33,345.60,33,28,1.772106,4.516019,33,5.000000,0,0.848485,0.000000,0.690798,0.933496
1614,835f0f7810c76831d6c7d24c7a646d4d,44,5130.00,40,31,4.745613,9.197471,40,3.475000,11,0.775000,0.275000,0.624966,0.876840
2039,a7f13822ceb966b076af67121f87b063,75,12477.64,75,45,3.011516,10.292731,74,3.337838,22,0.600000,0.297297,0.486878,0.703377
1032,54965bbe3e4f07ae045b90b0b8541f52,78,10961.30,72,41,11.926449,7.045856,68,3.058824,29,0.569444,0.426471,0.454431,0.677423
2004,a49928bcdf77c55c6d6e05e09a9b4ca5,98,8816.70,95,54,2.595666,6.235231,95,3.063158,37,0.568421,0.389474,0.468080,0.663443
333,1ca7077d890b907f89be8c954a02686a,115,13341.57,112,60,1.060501,5.840017,111,2.342342,68,0.535714,0.612613,0.443702,0.625358
2092,ad781527c93d00d89a11eecd9dcad7c1,44,6899.57,34,18,2.983480,7.860706,34,3.294118,12,0.529412,0.352941,0.367364,0.685487
2649,db2956745b3a8e9f3785c99f34b5d25e,47,8599.00,44,23,0.307163,6.461296,44,3.818182,9,0.522727,0.204545,0.379352,0.662452
1669,88460e8ebdecbfecb5f9601833981930,248,31546.55,224,116,3.786702,6.533762,223,3.493274,64,0.517857,0.286996,0.452672,0.582440
351,1da366cade6d8276e7d8beea7af5d4bf,43,6917.59,42,21,2.120031,6.634421,41,4.000000,7,0.500000,0.170732,0.355257,0.644743


,dispatch_late,review_count,mean_score,negative_rate
0,0,85596,4.193899,0.118896
1,1,8661,3.682023,0.24212


,mean,count
purchase_month,,
2016-09,0.5,2
2016-10,0.498141,269
2016-12,0.0,1
2017-01,0.219287,757
2017-02,0.174256,1647
2017-03,0.066899,2586
2017-04,0.072294,2310
2017-05,0.073157,3554
2017-06,0.077704,3153


mean  count
dispatch_late delivery_late                 
0             0              4.328565  80109
              1              2.270365   4579
              <NA>           2.013216    908
1             0              4.080000   6775
              1              2.267422   1765
              <NA>           2.033058    121

In [9]:
# [IA] registro 2
display(ranking_chart(metrics[metrics.eligible_orders.ge(30)]))
display(review_chart(comparison))
plt.close("all")

<Figure size 1000x500 with 1 Axes>

<Figure size 800x400 with 1 Axes>

## 7. Validação da reprodução em recortes reais
A amostra preserva itens e dimensões dos pedidos sorteados. Um recorte independente pode não possuir correspondência. A cobertura dos itens é indispensável para interpretar vendedor único.

In [10]:
# [IA] registro 3
o, i, p, s, r = raw
osmall = o.sample(20, random_state=4)
ismall = i[i.order_id.isin(osmall.order_id)]
small = preprocessar(osmall, ismall, p[p.product_id.isin(ismall.product_id)], s[s.seller_id.isin(ismall.seller_id)], r[r.order_id.isin(osmall.order_id)])
assert not small.duplicated(["order_id", "seller_id"]).any()
assert len(small) == len(ismall[["order_id", "seller_id"]].drop_duplicates())
print("Recorte real:", small.shape)
display(small[["order_id", "seller_id", "eligible_dispatch", "dispatch_late"]].head())

Recorte real: (19, 41)


,order_id,seller_id,eligible_dispatch,dispatch_late
0,003d9fc84ad902adf2265248b5ffe1a4,06a2c3af7b3aee5d69171b0e14f0ee87,True,0
1,046bac99a3d9ff45ef86ebcd710b972a,6560211a19b47992c3666cc44a7e94c0,True,0
2,09c0dd7d518cf413ce20facd85bedd5f,751bdc4d83a466c7206cd42e8f426b03,True,0
3,0a72bb193c65199cdd69574aac533a83,955fee9216a65b617aa5c0531780ce60,True,0
4,26b5d26aa210d44b8a3f2310d32643f6,2bf28e311bba65237358230cabac3f15,True,0


## 8. Campos de autoria humana
Respostas de negócio: preencher no painel e relatório. Justificativas: preencher diário de decisões. Conclusões: escrever pela equipe após validar os resultados. Não atribuir automaticamente estes resultados a revisão dos integrantes.